# 📡 Notebook 4: Server-Sent Events (SSE)

SSE is a powerful upgrade to long polling - the server can send **multiple messages** over a **single connection**! It's perfect for one-way real-time updates.

## Learning Objectives

By the end of this notebook, you'll understand:
- How SSE works (chunked transfer encoding)
- The SSE message format
- When to use SSE vs other methods
- How to build a real-time dashboard with SSE

## 🤔 What is Server-Sent Events?

With long polling, after each message, you need to make a **new request**. SSE keeps the connection open!

```
Long Polling:                      SSE:
                                   
Request  ──►                       Request  ──►
Response ◄── (message 1)           ◄── message 1 (chunked)
Request  ──►                       ◄── message 2 (chunked)
Response ◄── (message 2)           ◄── message 3 (chunked)
Request  ──►                       ◄── message 4 (chunked)
Response ◄── (message 3)           ...continues...

Multiple connections!              Single connection!
```

The magic: **chunked transfer encoding** - the server doesn't have to know the response size upfront!

## 📝 SSE Message Format

SSE has a simple text-based format:

```
data: {"message": "Hello!"}

data: {"message": "Another one!"}

event: custom_event
data: {"type": "notification"}

id: 42
data: {"message": "With an ID!"}

```

Each message ends with **two newlines**. Key fields:
- `data:` - The actual message content (required)
- `event:` - Custom event type (optional)
- `id:` - Message ID for reconnection (optional)
- `retry:` - Reconnection interval in ms (optional)

## 🛠️ Let's Build It!

### Step 1: Start the Server

Before continuing, start the server in a terminal:

```bash
cd patterns/real-time-updates/servers
python sse_server.py
```

You should see: `🚀 Starting SSE Server on port 5003`

In [1]:
# Verify the server is running
import requests

try:
    response = requests.get("http://localhost:5003/health", timeout=2)
    if response.status_code == 200:
        print("✅ SSE server is running!")
except requests.exceptions.ConnectionError:
    print("❌ Server is not running!")
    print("   Please start it with: python ../servers/sse_server.py")

✅ SSE server is running!


### Step 2: Create the SSE Client

In [2]:
import requests
import json
import threading
import time
from datetime import datetime

class SSEClient:
    """
    A client for consuming Server-Sent Events.
    """
    
    def __init__(self, url: str):
        self.url = url
        self.running = False
        self.last_id = None
        self.messages = []
    
    def parse_sse(self, line: str):
        """
        Parse a single SSE line.
        """
        if line.startswith("data:"):
            return ("data", line[5:].strip())
        elif line.startswith("event:"):
            return ("event", line[6:].strip())
        elif line.startswith("id:"):
            return ("id", line[3:].strip())
        return None
    
    def listen(self, duration: int = None, callback=None):
        """
        Listen for SSE events.
        
        Args:
            duration: How long to listen (None = forever)
            callback: Function to call for each message
        """
        self.running = True
        start_time = time.time()
        
        headers = {
            "Accept": "text/event-stream",
            "Cache-Control": "no-cache",
        }
        
        # Include last event ID for reconnection
        if self.last_id:
            headers["Last-Event-ID"] = str(self.last_id)
        
        try:
            # stream=True enables chunked reading
            response = requests.get(
                self.url,
                headers=headers,
                stream=True,
                timeout=None
            )
            
            current_event = None
            current_data = None
            current_id = None
            
            for line in response.iter_lines(decode_unicode=True):
                # Check if we should stop
                if not self.running:
                    break
                if duration and (time.time() - start_time) > duration:
                    break
                
                if line:
                    parsed = self.parse_sse(line)
                    if parsed:
                        field, value = parsed
                        if field == "data":
                            current_data = value
                        elif field == "event":
                            current_event = value
                        elif field == "id":
                            current_id = value
                            self.last_id = value
                
                else:
                    # Empty line = end of message
                    if current_data:
                        try:
                            data = json.loads(current_data)
                        except json.JSONDecodeError:
                            data = current_data
                        
                        message = {
                            "event": current_event or "message",
                            "data": data,
                            "id": current_id,
                            "received_at": datetime.now().isoformat()
                        }
                        
                        self.messages.append(message)
                        
                        if callback:
                            callback(message)
                        
                        # Reset for next message
                        current_event = None
                        current_data = None
                        current_id = None
                        
        except Exception as e:
            print(f"❌ SSE connection error: {e}")
        finally:
            self.running = False
    
    def stop(self):
        """Stop listening."""
        self.running = False

def send_message(user: str, text: str):
    """Send a message via the HTTP API."""
    try:
        response = requests.post(
            "http://localhost:5003/send",
            json={"user": user, "text": text},
            timeout=5
        )
        return response.status_code == 201
    except:
        return False

print("✅ SSE client created!")

✅ SSE client created!


## 🧪 Experiment: See SSE in Action!

In [3]:
# Let's see SSE in action with a simple listener

def on_message(message):
    """Callback for each received message."""
    event = message['event']
    data = message['data']
    
    if event == "connection":
        print(f"🔗 Connected: {data.get('message')}")
    elif event == "heartbeat":
        print(f"💓 Heartbeat received")
    elif event == "message":
        print(f"📬 Message: {data.get('user')}: {data.get('text')}")
    else:
        print(f"📨 {event}: {data}")

# Create client
client = SSEClient("http://localhost:5003/events")

# Send some messages while listening
def send_test_messages():
    time.sleep(1)
    print("\n📤 Sending test messages...")
    send_message("Alice", "Hello SSE!")
    time.sleep(1)
    send_message("Bob", "This is real-time!")
    time.sleep(1)
    send_message("Charlie", "Amazing!")

# Start message sender
sender = threading.Thread(target=send_test_messages)
sender.start()

# Listen for 5 seconds
print("🔄 Listening for SSE events...\n")
client.listen(duration=5, callback=on_message)

sender.join()
print("\n✅ Done! Check how all messages came over ONE connection!")

🔄 Listening for SSE events...

🔗 Connected: Welcome!



📤 Sending test messages...
📬 Message: Alice: Hello SSE!


📬 Message: Bob: This is real-time!


📬 Message: Charlie: Amazing!



✅ Done! Check how all messages came over ONE connection!


## 📊 SSE vs Long Polling: Burst Messages

Let's see how SSE handles burst messages better than long polling:

In [4]:
# Demonstrate SSE handling burst messages efficiently

def burst_test():
    """
    Send multiple messages in quick succession and see how SSE handles them.
    """
    print("📊 Burst Message Test")
    print("="*50)
    
    # Create fresh client
    client = SSEClient("http://localhost:5003/events")
    received_times = []
    
    def track_message(message):
        if message['event'] == 'message':
            received_times.append(time.time())
            print(f"📬 Received: {message['data'].get('text')}")
    
    # Send burst messages
    def send_burst():
        time.sleep(1)  # Wait for connection
        
        print("\n📤 Sending 5 messages with 100ms gaps...\n")
        send_times = []
        
        for i in range(5):
            send_times.append(time.time())
            send_message("Burst", f"Message {i+1}")
            time.sleep(0.1)  # 100ms between messages
        
        return send_times
    
    # Start in separate threads
    sender = threading.Thread(target=send_burst)
    sender.start()
    
    # Listen for messages
    client.listen(duration=4, callback=track_message)
    
    sender.join()
    
    print(f"\n📊 Results:")
    print(f"   Messages sent:     5")
    print(f"   Messages received: {len(received_times)}")
    
    if len(received_times) >= 2:
        gaps = [received_times[i+1] - received_times[i] 
                for i in range(len(received_times)-1)]
        avg_gap = sum(gaps) / len(gaps) * 1000
        print(f"   Avg gap between receives: {avg_gap:.1f}ms")
        print(f"\n💡 With long polling, each message would need a new request!")
        print(f"   With SSE, they all came over the SAME connection.")

burst_test()

📊 Burst Message Test



📤 Sending 5 messages with 100ms gaps...

📬 Received: Message 1
📬 Received: Message 2


📬 Received: Message 3
📬 Received: Message 4


📬 Received: Message 5



📊 Results:
   Messages sent:     5
   Messages received: 5
   Avg gap between receives: 163.9ms

💡 With long polling, each message would need a new request!
   With SSE, they all came over the SAME connection.


## 🔄 Automatic Reconnection

SSE has built-in reconnection support using the `id` and `Last-Event-ID` header:

In [5]:
# Demonstrate SSE reconnection concept

print("🔄 SSE Reconnection Concept")
print("="*50)
print("""
When the connection drops, browsers automatically reconnect!

How it works:

1. Server sends messages with IDs:
   id: 1
   data: {"message": "First"}
   
   id: 2
   data: {"message": "Second"}

2. Connection drops...

3. Client reconnects with header:
   Last-Event-ID: 2

4. Server sends only messages AFTER ID 2!

This means: NO MISSED MESSAGES! 🎉
""")

print("Our client stores the last ID:")
print(f"   client.last_id = {client.last_id}")
print("\nOn reconnection, it would send: Last-Event-ID: " + str(client.last_id))

🔄 SSE Reconnection Concept

When the connection drops, browsers automatically reconnect!

How it works:

1. Server sends messages with IDs:
   id: 1
   data: {"message": "First"}

   id: 2
   data: {"message": "Second"}

2. Connection drops...

3. Client reconnects with header:
   Last-Event-ID: 2

4. Server sends only messages AFTER ID 2!

This means: NO MISSED MESSAGES! 🎉

Our client stores the last ID:
   client.last_id = 45

On reconnection, it would send: Last-Event-ID: 45


## 🌐 Browser Support

Modern browsers have built-in SSE support via the `EventSource` API:

```javascript
// Browser JavaScript - it's this simple!
const eventSource = new EventSource('/events');

eventSource.onmessage = (event) => {
    console.log('Received:', event.data);
};

eventSource.onerror = () => {
    console.log('Connection lost, auto-reconnecting...');
};

// To close:
// eventSource.close();
```

The browser handles:
- Connection management
- Automatic reconnection
- Last-Event-ID tracking

## ✅ Advantages of SSE

1. **Single connection** - Multiple messages, one connection
2. **Built into browsers** - `EventSource` API
3. **Automatic reconnection** - Browser handles it
4. **Message IDs** - No missed messages
5. **Still HTTP** - Works with existing infrastructure
6. **Text-based** - Easy to debug

## ❌ Disadvantages

1. **One-way only** - Server → Client (no client → server)
2. **Proxy issues** - Some proxies buffer responses
3. **Browser limits** - ~6 connections per domain
4. **Text only** - Binary data needs encoding
5. **Monitoring** - Long-lived requests look odd in metrics

In [6]:
def check_sse_stats():
    try:
        response = requests.get("http://localhost:5003/stats")
        stats = response.json()
        
        print("📊 SSE Server Statistics")
        print("="*40)
        print(f"   Connected clients: {stats['connected_clients']}")
        print(f"   Messages sent:     {stats['messages_sent']}")
        print(f"   Buffer size:       {stats['buffer_size']}/{stats['buffer_max_size']}")
    except:
        print("❌ Could not fetch stats")

check_sse_stats()

📊 SSE Server Statistics
   Connected clients: 0
   Messages sent:     50
   Buffer size:       50/100


## 🔄 Deep Dive: Last-Event-ID and Reconnection

One of SSE's most powerful features is **automatic message recovery**. Here's how it works:

### The Problem: What happens when a connection drops?

```
Client connected ──────> receives msg 1, 2, 3
        |
   [Network drops]
        |
   (messages 4, 5 sent while disconnected)
        |
Client reconnects ────> HOW does it get messages 4, 5?
```

### The Solution: Message IDs and Last-Event-ID Header

1. **Server assigns IDs** to each message: `id: 123`
2. **Client tracks** the last received ID
3. **On reconnect**, client sends header: `Last-Event-ID: 123`
4. **Server replays** messages with ID > 123

This is why our updated server maintains a **message buffer**!

In [7]:
# Let's demonstrate Last-Event-ID reconnection!

import requests
import threading
import time

def demonstrate_reconnection():
    """
    This demonstrates how Last-Event-ID works:
    1. Send some messages (which get buffered)
    2. Simulate a client disconnecting  
    3. Send messages while client is "offline"
    4. Reconnect WITH Last-Event-ID header
    5. See the missed messages get replayed!
    """
    print("🎯 Last-Event-ID Demonstration")
    print("="*50)
    
    # Step 1: Check the current message buffer
    print("\n📦 Step 1: Check current message buffer")
    try:
        response = requests.get("http://localhost:5003/buffer")
        buffer_data = response.json()
        print(f"   Messages in buffer: {buffer_data['count']}")
        if buffer_data['messages']:
            last_msg = buffer_data['messages'][-1]
            print(f"   Last message ID: {last_msg['id']}")
            print(f"   Last message: '{last_msg['text']}'")
    except Exception as e:
        print(f"   ❌ Error: {e}")
        print("   Make sure the server is running!")
        return
    
    # Step 2: Send some new messages  
    print("\n📨 Step 2: Sending 3 messages to create a baseline...")
    for i in range(3):
        response = requests.post(
            "http://localhost:5003/send",
            json={"user": "demo", "text": f"Message {i+1} - online"}
        )
        msg = response.json()
        print(f"   Sent: '{msg['text']}' (ID: {msg['id']})")
        time.sleep(0.3)
    
    # Record the last ID we "saw"
    last_seen_id = msg['id']
    print(f"\n💾 Client remembers: Last-Event-ID = {last_seen_id}")
    
    # Step 3: Simulate being offline - send messages without receiving
    print("\n📴 Step 3: Client goes OFFLINE (simulated disconnect)")
    print("   Sending 3 more messages while client is 'offline'...")
    
    for i in range(3):
        response = requests.post(
            "http://localhost:5003/send",
            json={"user": "demo", "text": f"Message {i+1} - MISSED (sent while offline)"}
        )
        msg = response.json()
        print(f"   Server sent: '{msg['text']}' (ID: {msg['id']})")
        time.sleep(0.3)
    
    # Step 4: Reconnect with Last-Event-ID
    print(f"\n📱 Step 4: Client RECONNECTS with Last-Event-ID: {last_seen_id}")
    print("   Watch - the server should replay missed messages!\n")
    
    # Make a request WITH the Last-Event-ID header (simulating browser behavior)
    headers = {
        "Accept": "text/event-stream",
        "Last-Event-ID": str(last_seen_id)  # This is what browsers send!
    }
    
    # Read just a few events to see the replay
    response = requests.get(
        "http://localhost:5003/events",
        headers=headers,
        stream=True,
        timeout=5
    )
    
    print("   📥 Messages received on reconnect:")
    events_received = 0
    # Expected: 3 replayed missed messages + 1 connection event = 4.
    # After that the stream goes idle until the next heartbeat (~15s),
    # so we stop reading early and swallow any timeout.
    try:
        for line in response.iter_lines(decode_unicode=True):
            if line and line.startswith("data:"):
                data = line[5:].strip()
                print(f"      {data}")
                events_received += 1
                if events_received >= 4:
                    break
    except requests.exceptions.RequestException:
        pass  # idle-after-replay timeout is expected
    finally:
        response.close()
    
    print("\n" + "="*50)
    print("✅ Notice how the server replayed the 3 'MISSED' messages!")
    print("   This is the magic of Last-Event-ID - no messages lost!")

# Run the demonstration
demonstrate_reconnection()


🎯 Last-Event-ID Demonstration

📦 Step 1: Check current message buffer
   Messages in buffer: 50
   Last message ID: 50
   Last message: 'Message 5'

📨 Step 2: Sending 3 messages to create a baseline...
   Sent: 'Message 1 - online' (ID: 51)


   Sent: 'Message 2 - online' (ID: 52)


   Sent: 'Message 3 - online' (ID: 53)



💾 Client remembers: Last-Event-ID = 53

📴 Step 3: Client goes OFFLINE (simulated disconnect)
   Sending 3 more messages while client is 'offline'...
   Server sent: 'Message 1 - MISSED (sent while offline)' (ID: 54)


   Server sent: 'Message 2 - MISSED (sent while offline)' (ID: 55)


   Server sent: 'Message 3 - MISSED (sent while offline)' (ID: 56)



📱 Step 4: Client RECONNECTS with Last-Event-ID: 53
   Watch - the server should replay missed messages!

   📥 Messages received on reconnect:
      {"id": 54, "user": "demo", "text": "Message 1 - MISSED (sent while offline)", "timestamp": "2026-04-19T11:20:43.364525"}
      {"id": 55, "user": "demo", "text": "Message 2 - MISSED (sent while offline)", "timestamp": "2026-04-19T11:20:43.737242"}
      {"id": 56, "user": "demo", "text": "Message 3 - MISSED (sent while offline)", "timestamp": "2026-04-19T11:20:44.114486"}
      {"type": "connected", "message": "Welcome!", "recovered_from": 53}

✅ Notice how the server replayed the 3 'MISSED' messages!
   This is the magic of Last-Event-ID - no messages lost!


### 🧠 Understanding the Flow

Here's what happened in the demo above:

```
Timeline:
─────────────────────────────────────────────────────────────────
 Online Phase          │ Offline Phase         │ Reconnection
─────────────────────────────────────────────────────────────────
 Msg 1 ✓ received      │                       │
 Msg 2 ✓ received      │                       │
 Msg 3 ✓ received      │                       │
 (last_seen_id = 3)    │                       │
                       │ Msg 4 → buffer only   │
                       │ Msg 5 → buffer only   │ 
                       │ Msg 6 → buffer only   │
                       │                       │ Connect with Last-Event-ID: 3
                       │                       │ Server replays: 4, 5, 6 ✓
─────────────────────────────────────────────────────────────────
```

### Key Points:

1. **Buffer is finite** - Our server keeps last 100 messages. If you're offline too long, old messages are lost!
2. **Browser handles this** - With `EventSource`, you don't write any reconnection code
3. **Server must implement it** - Not all SSE servers support Last-Event-ID!

In [8]:
# View the current message buffer (debug endpoint)

def view_message_buffer():
    """
    Our server has a /buffer endpoint to see what's stored.
    This is what the server uses to replay on reconnection.
    """
    print("📦 Current Message Buffer")
    print("="*50)
    
    try:
        response = requests.get("http://localhost:5003/buffer")
        data = response.json()
        
        print(f"Total messages buffered: {data['count']}")
        print(f"Buffer capacity: 100 messages")
        print()
        
        if data['messages']:
            print("Last 5 messages in buffer:")
            for msg in data['messages'][-5:]:
                print(f"  ID {msg['id']:3d}: [{msg['user']}] {msg['text']}")
        else:
            print("Buffer is empty")
            
        # Also show stats
        stats = requests.get("http://localhost:5003/stats").json()
        print(f"\n📊 Server Stats:")
        print(f"   Connected clients: {stats['connected_clients']}")
        print(f"   Total messages sent: {stats['messages_sent']}")
        
    except Exception as e:
        print(f"❌ Error: {e}")
        print("Make sure the SSE server is running!")

view_message_buffer()

📦 Current Message Buffer
Total messages buffered: 56
Buffer capacity: 100 messages

Last 5 messages in buffer:
  ID  52: [demo] Message 2 - online
  ID  53: [demo] Message 3 - online
  ID  54: [demo] Message 1 - MISSED (sent while offline)
  ID  55: [demo] Message 2 - MISSED (sent while offline)
  ID  56: [demo] Message 3 - MISSED (sent while offline)

📊 Server Stats:
   Connected clients: 0
   Total messages sent: 56


## 🎯 When to Use SSE

SSE is perfect when:

| Use Case | Why SSE Works |
|----------|---------------|
| Live dashboards | One-way updates, high frequency |
| Stock tickers | Constant updates, read-only |
| AI chat responses | Streaming tokens as they're generated |
| Social feeds | New posts pushed to users |
| Notifications | Server pushes alerts to clients |

### Don't use SSE when:

- You need **bidirectional** communication (use WebSocket)
- You need **binary data** efficiently (use WebSocket)
- You're behind problematic **proxies** (test first!)

## 🔧 Real-World: AI Chat Streaming

A very popular use case for SSE today is AI chat applications (like ChatGPT) that stream tokens as they're generated:

In [9]:
# Simulate AI token streaming

def simulate_ai_streaming():
    """
    Simulate how AI chat apps stream tokens via SSE.
    """
    print("🤖 Simulating AI Response Streaming")
    print("="*50)
    print("\nAI response streaming word by word:\n")
    
    # Simulated AI response
    response_text = "Hello! I'm an AI assistant. I can help you with programming, answer questions, and have conversations. How can I assist you today?"
    words = response_text.split()
    
    print("User: What can you do?")
    print("\nAI: ", end="", flush=True)
    
    # Stream each word (simulating token-by-token generation)
    for word in words:
        print(word + " ", end="", flush=True)
        time.sleep(0.1)  # Simulate generation time
    
    print("\n\n" + "="*50)
    print("\n💡 In real AI apps, each word is sent as an SSE event:")
    print("")
    print('   event: token')
    print('   data: {"text": "Hello!"}')
    print('   ')
    print('   event: token')
    print('   data: {"text": "I\'m"}')
    print('   ')
    print('   event: done')
    print('   data: {"finished": true}')

simulate_ai_streaming()

🤖 Simulating AI Response Streaming

AI response streaming word by word:

User: What can you do?

AI: 

Hello! 

I'm 

an 

AI 

...

I 

assist 

you 

today? 




💡 In real AI apps, each word is sent as an SSE event:

   event: token
   data: {"text": "Hello!"}
   
   event: token
   data: {"text": "I'm"}
   
   event: done
   data: {"finished": true}


## 🧪 Quick Quiz

1. **What HTTP feature makes SSE possible?**

2. **You're building a live sports score dashboard. Would SSE be a good choice?**

3. **A user is chatting with friends. Would you use SSE or WebSocket?**

In [10]:
# Quiz answers

print("📝 Quiz Answers")
print("="*50)
print("")
print("1. CHUNKED TRANSFER ENCODING!")
print("   The server doesn't send Content-Length, so it can")
print("   keep sending data chunks indefinitely.")
print("")
print("2. YES! SSE is perfect for live dashboards.")
print("   Users only receive updates, they don't send data.")
print("")
print("3. WEBSOCKET! Chat requires bidirectional communication.")
print("   Users need to both send AND receive messages.")
print("   SSE is one-way only (server → client).")

📝 Quiz Answers

1. CHUNKED TRANSFER ENCODING!
   The server doesn't send Content-Length, so it can
   keep sending data chunks indefinitely.

2. YES! SSE is perfect for live dashboards.
   Users only receive updates, they don't send data.

3. WEBSOCKET! Chat requires bidirectional communication.
   Users need to both send AND receive messages.
   SSE is one-way only (server → client).


## 📊 Comparison Table

| Feature | Simple Polling | Long Polling | SSE |
|---------|---------------|--------------|-----|
| Latency | High (interval) | Low | Very Low |
| Connection | New each time | New each response | Persistent |
| Direction | Request-Response | Request-Response | Server → Client |
| Burst handling | ✅ Consistent | ⚠️ Adds latency | ✅ Efficient |
| Browser support | ✅ Native | ✅ Native | ✅ EventSource API |
| Reconnection | Manual | Manual | Automatic |
| Complexity | Very Low | Low | Medium |

## 📚 Summary

### What We Learned:

1. **SSE** = Server sends multiple messages over ONE connection
2. Uses **chunked transfer encoding** (no Content-Length)
3. Simple **text format** with data, event, id fields
4. **Browser built-in** support via `EventSource`
5. **Automatic reconnection** with Last-Event-ID
6. **One-way only** - server to client

### Interview Tips:

> "SSE is perfect when I need server-to-client updates efficiently. It's more efficient than long polling because all messages come over one connection, and browsers handle reconnection automatically. If I needed bidirectional communication, I'd use WebSocket instead."

### Next Up: WebSockets

In the next notebook, we'll explore **WebSockets** - the full-duplex solution for bidirectional real-time communication!